# Step 1 — Collect the raw data from the NSE website
**Project:** Indian Equity Market Analysis using NSE Data — liquidity, investor conviction (delivery %) and sector performance

Run this notebook **once** in Google Colab (Runtime → Run all). It downloads two official files from the National Stock Exchange of India and saves them **unchanged**:
- `nse_bhavcopy_raw.csv` — the **Full Bhavcopy with delivery data**: one row per security per trading day (price, volume, turnover, trades, delivered quantity and delivery %)
- `nifty500_industry_raw.csv` — NSE's official **Nifty 500 list** with each company's industry

Both files download automatically at the end. Upload them to the chat (and to your GitHub repo) for Step 2: the analysis notebook.

**If the NSE website blocks Colab** (it sometimes does), the notebook prints a manual fallback at the end: open the single-day link in your phone browser instead.

In [ ]:
import io, time, datetime as dt
import requests
import pandas as pd

## Download the daily Full Bhavcopy (with delivery data)
Window: **the whole of October 2026 (1 Oct → 31 Oct)**. Weekends, exchange holidays and days that are not published yet are skipped automatically. If NSE answers a holiday request with the *previous* session's file (it does), the notebook notices that the file's own date does not match and skips it, so no trading day is stored twice.

In [ ]:
START, END = dt.date(2026, 10, 1), dt.date(2026, 10, 31)
BHAV_URL = "https://nsearchives.nseindia.com/products/content/sec_bhavdata_full_{d}.csv"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept": "text/csv,application/csv,text/plain,*/*",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.nseindia.com/all-reports",
}
session = requests.Session()
session.headers.update(HEADERS)
try:
    session.get("https://www.nseindia.com", timeout=15)      # warm-up: lets NSE set its cookies
except Exception as err:
    print("warm-up skipped:", err)

frames, got, skipped = [], [], []
day = START
while day <= END:
    if day.weekday() < 5:                                    # Mon-Fri only
        url = BHAV_URL.format(d=day.strftime("%d%m%Y"))
        done = False
        for attempt in range(3):
            try:
                r = session.get(url, timeout=30)
                if r.status_code == 200 and len(r.content) > 10_000:
                    df = pd.read_csv(io.StringIO(r.text), skipinitialspace=True)
                    df.columns = df.columns.str.strip()
                    file_day = pd.to_datetime(df["DATE1"].iloc[0], format="%d-%b-%Y").date()
                    if file_day != day:                      # holiday: NSE re-serves the previous session's file -> skip it
                        break
                    frames.append(df); got.append(day); done = True
                    break
                if r.status_code == 404:                     # holiday or not published yet
                    break
            except Exception as err:
                print("   retry", day, "->", err)
            time.sleep(2 * (attempt + 1))
        if not done:
            skipped.append(day)
        time.sleep(0.6)
    day += dt.timedelta(days=1)

print(f"Trading sessions downloaded: {len(got)}   |   weekdays skipped (holiday / not published yet / blocked): {len(skipped)}")
if got:
    print("First session:", got[0].strftime("%d %b %Y"), "| last session:", got[-1].strftime("%d %b %Y"))
    print("Skipped weekdays:", [d.strftime("%d %b") for d in skipped])
    if got[-1] < dt.date(2026, 10, 30):
        print("NOTE: October is not finished yet - run this notebook again after 7 PM IST on 30 Oct to get the full month.")
if frames:
    bhav = pd.concat(frames, ignore_index=True)
    bhav.to_csv("nse_bhavcopy_raw.csv", index=False)
    print("Saved nse_bhavcopy_raw.csv ->", bhav.shape)
else:
    bhav = pd.DataFrame()
    print("No bhavcopy files could be downloaded automatically - use the manual fallback at the end.")

## Download the official Nifty 500 list (company → industry)

In [ ]:
NIFTY_URLS = [
    "https://nsearchives.nseindia.com/content/indices/ind_nifty500list.csv",
    "https://www.niftyindices.com/IndexConstituent/ind_nifty500list.csv",
]
nifty = pd.DataFrame()
for url in NIFTY_URLS:
    try:
        r = session.get(url, timeout=30)
        if r.status_code == 200 and len(r.content) > 5_000:
            nifty = pd.read_csv(io.StringIO(r.text), skipinitialspace=True)
            nifty.columns = nifty.columns.str.strip()
            nifty.to_csv("nifty500_industry_raw.csv", index=False)
            print("Saved nifty500_industry_raw.csv ->", nifty.shape, "| source:", url)
            break
    except Exception as err:
        print("   failed:", url, "->", err)
if nifty.empty:
    print("Could not fetch the Nifty 500 list automatically - download it manually (see fallback below).")

## Check and download

In [ ]:
if not bhav.empty:
    print("Columns:", list(bhav.columns))
    print("Rows per day (first 5):")
    print(bhav.groupby("DATE1").size().head().to_string())
    print("Series mix:", bhav["SERIES"].value_counts().head(6).to_dict())

try:
    from google.colab import files
    for f in ["nse_bhavcopy_raw.csv", "nifty500_industry_raw.csv"]:
        try:
            files.download(f)
        except Exception as err:
            print("could not download", f, "->", err)
except Exception:
    print("Not running in Colab - the files are saved in the working folder.")

print("""
MANUAL FALLBACK (if anything above failed) - open these in your phone/laptop browser and save the files:
 1) One day of the Full Bhavcopy (after ~6:30 PM IST on a trading day; edit the date as DDMMYYYY):
    https://nsearchives.nseindia.com/products/content/sec_bhavdata_full_21102026.csv
 2) Nifty 500 list (click 'Download List of Nifty 500 Stocks (.csv)'):
    https://www.nseindia.com/static/products-services/indices-nifty500-index
""")